# Day 13 — Properties, Inheritance, ABCs

> ⚠️ **Why this matters.** Two more OOP tools: **properties** (computed attributes), and **inheritance** (one class extends another). Both are easy to overuse — today you learn when each pays off and when it costs you more than it saves.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/13-properties-inheritance.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min refactor + 45 min quiz.

- [ ] You can define `@property` for computed attributes
- [ ] You can write a subclass and override methods
- [ ] You know when to inherit vs compose
- [ ] You can use abstract base classes (ABCs) for interfaces
- [ ] Your `Word` has computed properties; you've prototyped a `QuizMode` hierarchy

## 1. Properties — computed attributes

In [ ]:
from dataclasses import dataclass

@dataclass
class Word:
    word: str
    ipa: str = ''
    thai: str = ''

    @property
    def is_long(self) -> bool:
        return len(self.word) >= 8

    @property
    def display(self) -> str:
        return f'{self.word} ({self.ipa})' if self.ipa else self.word

w = Word('thorough', '/ˈθʌrə/')
print(w.is_long)     # True — looks like an attribute, computed on access
print(w.display)     # thorough (/ˈθʌrə/)

**Why properties?** Read like attributes, behave like methods. Useful when:

- The value is *derived* from other attributes (no need to store)
- You want a public API that hides implementation
- You might add validation later without changing callers

**Don't overuse.** If it's complex or has side effects, make it a regular method. Surprise behavior on `obj.x` access is bad.

## 2. Inheritance basics

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    def speak(self) -> str:
        return 'Some sound'

class Dog(Animal):
    def speak(self) -> str:
        return 'Woof'

class Cat(Animal):
    def speak(self) -> str:
        return 'Meow'

for a in [Dog('Rex'), Cat('Bella')]:
    print(f'{a.name}: {a.speak()}')

**Anatomy:**

- `class Dog(Animal):` — Dog inherits from Animal
- Dog has Animal's `__init__` automatically (didn't define its own)
- Dog **overrides** `speak()` with its own version
- `isinstance(Dog('Rex'), Animal)` → True

> ⚠️ **The deep-inheritance trap.** If you find yourself doing `class C(B)`, `class B(A)`, `class A(Base)` — 4 levels deep — stop. That's hard to read. Prefer **composition** (one class HAS another) over **inheritance** (one class IS another).

## 3. `super()` — call the parent's method

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)   # call Animal's __init__
        self.breed = breed

d = Dog('Rex', 'Beagle')
print(d.name, d.breed)

## 4. Abstract base classes (ABCs)

In [ ]:
from abc import ABC, abstractmethod

class QuizMode(ABC):
    @abstractmethod
    def next_word(self, words: list) -> dict:
        ...

    @abstractmethod
    def grade(self, entry: dict, answer: str) -> bool:
        ...

class RandomQuiz(QuizMode):
    def next_word(self, words):
        import random; return random.choice(words)
    def grade(self, entry, answer):
        return entry['ipa'] == answer.strip()

# QuizMode()             # TypeError — can't instantiate abstract
q = RandomQuiz()
print(q.next_word([{'word': 'a', 'ipa': '/a/'}]))

**What ABCs give you:**

- A contract: "any subclass must implement these methods"
- Can't instantiate an incomplete subclass
- Documents the interface

Tomorrow's design choice: should `QuizMode` be an ABC or just a regular class? Both work; ABC is stricter.

## 5. Composition over inheritance

**Composition** = a class CONTAINS another class, instead of inheriting from it.

When you'd say "X is a Y" → inherit.
When you'd say "X has a Y" → compose.

Examples:

- Dog IS an Animal → inherit
- Quiz HAS a Word list → compose
- Car HAS an Engine → compose
- Engine IS a thing → don't make Car inherit from Engine

**Composition is usually right.** Inheritance adds coupling. Once two classes are stuck in a hierarchy, refactoring is painful.

## End-of-day mini-project — `Word` properties + `QuizMode` hierarchy

> 🎯 **Today's piece:** add 2-3 properties to `Word`, prototype a quiz-mode class hierarchy.

### Word additions

Add to `Word`:

- `is_long` — `len(self.word) >= 8`
- `has_ipa` — bool
- `display` — formatted summary string
- (No setters needed; these are computed.)

### QuizMode prototype

Create `src/english_helper/quiz_modes.py`:

```python
from abc import ABC, abstractmethod

class QuizMode(ABC):
    @abstractmethod
    def next_word(self, words: list[Word]) -> Word: ...

    @abstractmethod
    def grade(self, w: Word, answer: str) -> bool: ...

class RandomQuiz(QuizMode): ...   # random.choice
class WeakestFirst(QuizMode): ...  # picks the word you got wrong most
class SequentialQuiz(QuizMode): ...  # in order, one at a time
```

Pick 2 of the 3 to fully implement. The third can be a stub.

## Connect to the project

> 🎯 **Tomorrow (Day 14):** big refactor day — restructure english-helper around classes. No new concepts; you apply what you've learned.

**Quiz:** [13-properties-inheritance-quiz.ipynb](13-properties-inheritance-quiz.ipynb)